# GeoLife — Behavior EDA Mentor Demo v2 (Charts + User Maps)

**Mục tiêu:** gom toàn bộ EDA mới sau Home/Office baseline thành một narrative duy nhất để demo với mentor.

Notebook này không scan lại raw GeoLife. Nó reuse aggregate/private caches đã được validate từ:

- 03a — behavior-first EDA;
- 03b — mobile/distributed-mobility hypothesis audit;
- 03b.1 — observation-support-controlled audit.

Câu chuyện chính:

Frozen CP1 / CP2 baseline
→ mentor challenge fixed-office / fixed-schedule assumptions
→ 03a audit population heterogeneity
→ phát hiện robust 23-user high-mobility, multiple-anchor cohort
→ 03b challenge mobile/distributed-work hypothesis
→ 03b.1 equalize observation exposure
→ final interpretation: mobility-complexity cohort, chưa phải mobile-work semantic regime.

**Privacy:** notebook chỉ hiển thị aggregate outputs, không raw identifiers, coordinates hoặc case traces.

**Production rule:** notebook này không thay đổi frozen Home/Office semantics.

### Visual demo policy

GeoLife là public research dataset và notebook này dùng cho side-project / mentor review nội bộ. Vì vậy raw GeoLife `user_id` và spatial coordinates được hiển thị có chủ đích trong selected case maps.

Map/case chỉ để giải thích mechanics và behavioral pattern; population conclusion vẫn đến từ aggregate/sensitivity/bootstrap evidence. User ID không được diễn giải thành occupation / HOME / OFFICE ground truth.



## 0. Setup — reuse Modal Volume caches

Execution pattern giống các notebook EDA hiện tại:

- Modal Volume: /mnt/geolife-data
- Repo: /tmp/geolife
- Branch mặc định: eda/03b1-support-controlled-audit

Caches:

- /mnt/geolife-data/cache/03a_user_behavior_deep_dive
- /mnt/geolife-data/cache/03b_mobile_work_hypothesis
- /mnt/geolife-data/cache/03b1_observation_support_controlled

Nếu cache đã tồn tại, notebook chỉ load JSON/CSV và render aggregate tables/charts.


In [ ]:
from pathlib import Path
import json
import os
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get(
    "GEOLIFE_REPO_BRANCH",
    "eda/03b1-support-controlled-audit",
)
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
VOLUME_ROOT = Path("/mnt/geolife-data")

CACHE_03A = VOLUME_ROOT / "cache" / "03a_user_behavior_deep_dive"
CACHE_03B = VOLUME_ROOT / "cache" / "03b_mobile_work_hypothesis"
CACHE_03B1 = VOLUME_ROOT / "cache" / "03b1_observation_support_controlled"

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )

def load_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing cache: {path}. Run the corresponding audit first."
        )
    return json.loads(path.read_text(encoding="utf-8"))

def load_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing cache: {path}. Run the corresponding audit first."
        )
    return pd.read_csv(path)

ensure_repo()
os.chdir(REPO_DIR)

summary_03a = load_json(CACHE_03A / "summary.json")
summary_03b = load_json(CACHE_03B / "summary.json")
summary_03b1 = load_json(CACHE_03B1 / "summary.json")

print("Repo branch:", REPO_BRANCH)
print("Loaded 03a / 03b / 03b.1 aggregate caches.")


## 0.1 User-level visual layer

Bản v2 bổ sung selected user diagnostics giống tinh thần notebook 02a:

```text
aggregate evidence
+ deterministic user cases
+ interactive spatial maps
+ user-level temporal / route diagnostics
```

Phần này không scan raw trajectory files. Nó dùng frozen CP1 stay cache + 03a/03b user-level caches đã có.


In [ ]:
# User-level visualization layer.
from importlib.util import module_from_spec, spec_from_file_location
import sys

try:
    import timezonefinder  # noqa: F401
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "timezonefinder==9.0.0"],
        check=True,
    )

try:
    import folium
    from folium import plugins
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "folium>=0.15,<1"],
        check=True,
    )
    import folium
    from folium import plugins

BASE03A_PATH = REPO_DIR / "analysis/03a_user_behavior_deep_dive.py"

def load_module(name, path):
    spec = spec_from_file_location(name, path)
    assert spec and spec.loader
    module = module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

base03a = load_module("behavior_03a_mentor_demo", BASE03A_PATH)

FEATURES_03A = CACHE_03A / "user_behavior_features.csv"
STAYS_CACHE = CACHE_03A / "stays_baseline_v1.pkl"
POINT_DAY_CACHE = CACHE_03A / "cleaned_point_daily_metrics.pkl"
GROUP_MEMBERSHIP = CACHE_03B / "group_membership.csv"
MATCHED_CONTROLS = CACHE_03B / "matched_controls.csv"
ROUTE_STRUCTURE = CACHE_03B / "route_structure.csv"

features_user = pd.read_csv(FEATURES_03A, dtype={"user_id": str})
stays_user = pd.read_pickle(STAYS_CACHE)
point_days_raw = pd.read_pickle(POINT_DAY_CACHE)
membership_user = pd.read_csv(GROUP_MEMBERSHIP, dtype={"user_id": str, "group": str})
matched_user = pd.read_csv(
    MATCHED_CONTROLS,
    dtype={"candidate_user_id": str, "control_user_id": str},
)
route_user = pd.read_csv(ROUTE_STRUCTURE, dtype={"user_id": str, "group": str})

clustered_stays, location_summary = base03a.cluster_behavior_locations(
    stays_user,
    200.0,
)
point_days_behavior = base03a._point_days_for_behavior(point_days_raw)

MAP_EXPORT_DIR = VOLUME_ROOT / "exports" / "behavior_mentor_demo_maps"
MAP_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Loaded user-level caches")
print("stays:", f"{len(stays_user):,}")
print("feature users:", len(features_user))
print("03b memberships:", membership_user["group"].value_counts().to_dict())
print("map export dir:", MAP_EXPORT_DIR)


# Part I — Frozen baseline

Trước khi làm EDA mới, upstream contract phải giữ nguyên.

Frozen CP1:

- same_second_radius_m = 10 m
- max_gap_s = 300 s
- hard_speed_guard_kmh = 1200 km/h
- stay distance threshold = 200 m
- minimum dwell = 1200 s

Full release đã reconcile:

- 182 release users;
- 136 users có ít nhất một CP1 stay;
- 5,821 stays;
- frozen CP2 v1 comparator emit 27 HOME và 16 OFFICE.

**Lý do freeze:** nếu vừa thay cleaning/stay detection vừa làm behavior EDA, downstream difference có thể chỉ là artifact của upstream change.


In [ ]:
recon = summary_03a["reconciliation"]
emissions = summary_03a["comparator_emissions"]

display(
    pd.DataFrame(
        {
            "metric": [
                "Release users",
                "Users with CP1 stays",
                "Detected CP1 stays",
                "Frozen HOME emitted",
                "Frozen OFFICE emitted",
            ],
            "value": [
                recon["release_users"],
                recon["stay_users"],
                recon["stays"],
                emissions["HOME"],
                emissions["OFFICE"],
            ],
        }
    )
)


# Part II — 03a Behavior-first EDA

## 1. Mentor challenge

Home/Office baseline ban đầu dùng fixed behavioral windows:

- HOME: night dwell;
- OFFICE: weekday 09–17 dwell.

Mentor challenge:

- không phải ai cũng có fixed office;
- user có thể có nhiều recurring locations;
- schedule có thể khác 09–17;
- outlier có thể là valid behavior;
- work regime có thể ảnh hưởng mobility.

Thay vì sửa heuristic ngay, 03a hỏi trước:

**Behavior của user trong GeoLife thực sự heterogeneous đến mức nào?**


## 2. Coverage funnel — abstention có thể chỉ là thiếu support

Không emit label không đồng nghĩa behavior lạ.

03a tách:

release user
→ có CP1 stay hay không
→ có đủ schedule support hay không.

Đây là bước cần thiết trước khi diễn giải behavioral pattern.


In [ ]:
coverage = summary_03a["coverage"]

funnel = pd.DataFrame(
    {
        "stage": ["Release", "CP1 stay users", "Schedule-supported"],
        "users": [
            coverage["release_users"],
            coverage["stay_users"],
            coverage["schedule_supported_users"],
        ],
    }
)
display(funnel)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(funnel["stage"], funnel["users"])
ax.set_ylabel("Users")
ax.set_title("03a — Evidence coverage funnel")
plt.tight_layout()
plt.show()


Measured result:

- 182 release users;
- 136 users có CP1 stays;
- 46 schedule-supported;
- 90 stay-users vẫn chưa đủ schedule support.

**Không được suy ra:** 90 users đó có schedule bất thường. Chỉ biết evidence hiện tại chưa đủ mạnh.


## 3. Spatial heterogeneity — fixed-anchor assumption có đủ không?

03a audit recurring-anchor structure tại 200 m để tránh assume rằng user chủ yếu chỉ có Home + Office.


In [ ]:
anchors = summary_03a["anchors"]

anchor_table = pd.DataFrame(
    {
        "anchor_class": [
            "Dominant",
            "Two-anchor",
            "Multiple-anchor",
            "No stable anchor",
        ],
        "users": [
            anchors["dominant_anchor"],
            anchors["two_anchor"],
            anchors["multiple_anchor"],
            anchors["no_stable_anchor"],
        ],
    }
)
display(anchor_table)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(anchor_table["anchor_class"], anchor_table["users"])
ax.set_ylabel("Users with CP1 stays")
ax.set_title("03a — Recurring-anchor heterogeneity at 200 m")
plt.tight_layout()
plt.show()


Kết quả trên 136 stay-users:

- 19 dominant-anchor;
- 13 two-anchor;
- 72 multiple-anchor;
- 32 no-stable-anchor.

**Decision:** spatial behavior đa dạng hơn fixed Home+Office view; semantic layer không nên assume mọi user chỉ cần hai anchors.


## 4. Schedule personalization — vì sao chưa dùng JSD để thay fixed windows?

Một redesign tự nhiên là personalize work schedule theo user.

Nhưng trước khi làm vậy, cần chứng minh within-user schedule ổn định hơn null/reference.


In [ ]:
cal = summary_03a["schedule_calibration"]

display(
    pd.DataFrame(
        {
            "comparison": [
                "Within-user weekly JSD",
                "Between-user JSD",
                "Shuffled-week null",
            ],
            "median": [
                cal["within_user"]["median"],
                cal["between_user"]["median"],
                cal["shuffled_week"]["median"],
            ],
            "q1": [
                cal["within_user"]["q1"],
                cal["between_user"]["q1"],
                cal["shuffled_week"]["q1"],
            ],
            "q3": [
                cal["within_user"]["q3"],
                cal["between_user"]["q3"],
                cal["shuffled_week"]["q3"],
            ],
        }
    )
)


Measured result:

- within-user weekly JSD median = 1.000;
- between-user median = 0.904;
- shuffled-week null = 1.000.

**Decision:** current JSD check không establish personalized schedule structure.

Vì vậy không dùng JSD để personalize 09–17, không tạo user-specific OFFICE window, và không dùng nó làm v2 cutoff.

Đây là negative result quan trọng: EDA loại bỏ một redesign chưa đủ evidence.


## 5. Mobility hypothesis xuất hiện từ đâu?

03a đo cleaned-point mobility trên usable temporal-profile days:

- distance/day;
- movement-duration proxy/day;
- boundary count/day.

Đây là movement proxies, không phải true travel distance.

Trong audit xuất hiện một cohort 23 users:

- 23/23 multiple-anchor;
- 23/23 frozen OFFICE-abstained;
- 0/23 OFFICE-emitted;
- chỉ 2/23 có repeatable exact motif.

Tên mobile-work-like ở đây chỉ là descriptive wrapper, chưa phải occupation inference.


In [ ]:
mobility = summary_03a["mobility"]
overlap = summary_03a["candidate_overlap"]["mobile_work_like_intersections"]
counts = summary_03a["candidate_overlap"]["dimension_counts"]

display(
    pd.DataFrame(
        {
            "metric": [
                "Cleaned distance (km/day)",
                "Movement-duration proxy (h/day)",
                "Boundary count (/day)",
            ],
            "median": [
                mobility["distance_per_usable_day_km"]["median"],
                mobility["movement_proxy_per_usable_day_h"]["median"],
                mobility["boundary_count_per_usable_day"]["median"],
            ],
            "q1": [
                mobility["distance_per_usable_day_km"]["q1"],
                mobility["movement_proxy_per_usable_day_h"]["q1"],
                mobility["boundary_count_per_usable_day"]["q1"],
            ],
            "q3": [
                mobility["distance_per_usable_day_km"]["q3"],
                mobility["movement_proxy_per_usable_day_h"]["q3"],
                mobility["boundary_count_per_usable_day"]["q3"],
            ],
        }
    ).round(2)
)

display(
    pd.DataFrame(
        {
            "measure": [
                "Mobile-work-like candidates",
                "Multiple-anchor",
                "Repeatable exact motif",
                "Frozen OFFICE emitted",
                "Frozen OFFICE abstained",
            ],
            "users": [
                counts["mobile_work_like"],
                overlap["multiple_anchor"],
                overlap["motif_repeatable"],
                overlap["OFFICE_emitted"],
                overlap["OFFICE_abstained"],
            ],
        }
    )
)


# Part III — 03b Hypothesis audit

Question:

**Nhóm 23 users có phải distinct distributed/mobile-work-like behavioral regime hay chỉ là artifact của wrapper, support hoặc travel?**

Thiết kế:

- Group A = frozen 23 candidates;
- Group B = matched OFFICE-abstained controls;
- Group C = frozen OFFICE-emitted comparator.

Anti-circularity rule:

Không dùng lại chính construction features để validate hypothesis.

Independent evidence ưu tiên:

- route/transition structure;
- transport modes khi có labels;
- sensitivity;
- negative controls.


In [ ]:
groups = summary_03b["groups"]
display(
    pd.DataFrame(
        {
            "group": ["A candidate", "B matched control", "C OFFICE comparator"],
            "users": [groups["A"], groups["B"], groups["C"]],
        }
    )
)

sens = load_csv(CACHE_03B / "sensitivity.csv")
display(sens)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(sens["variant"], sens["jaccard"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Jaccard vs frozen 23")
ax.set_title("03b — Candidate robustness")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()


Sensitivity result:

- anchor 100 m → same 23;
- anchor 300 m → same 23;
- mobility threshold ±10% chỉ đổi khoảng 1 user;
- support ±1 weekday không đổi cohort.

**Decision:** cohort robust với tested thresholds.

Nhưng robust definition không đồng nghĩa validated semantics.


## 6. Matching confound

03b match đủ A/B pairs nhưng aggregate observation support vẫn lệch.

Nếu A được observe nhiều hơn, ta có thể thấy nhiều transitions/entropy hơn chỉ vì coverage.

Vì vậy route difference ở 03b chưa được đọc như causal/independent evidence.


In [ ]:
support = load_csv(CACHE_03B / "support_balance.csv")
support_groups = support[support["group"].isin(["A", "B", "C"])].copy()

display(
    support_groups[
        [
            "group",
            "active_days_median",
            "usable_temporal_days_median",
            "cp1_stay_count_median",
        ]
    ]
)

route_03b = pd.DataFrame(
    [{"group": g, **values} for g, values in summary_03b["transition_summary"].items()]
)
display(route_03b.round(3))

mode_03b = pd.DataFrame(summary_03b["transport_mode"])
display(
    mode_03b[
        [
            "group",
            "label_users",
            "matched_users",
            "median_matched_duration_h_per_user",
            "median_matched_distance_km_per_user",
            "motorized_mode_distance_share",
        ]
    ].round(3)
)


03b result:

- candidate robustness: strong;
- route complexity difference: suggestive;
- recurrent route difference: weak;
- transport coverage: sparse;
- A/B observation balance: problematic.

Research decision: **mixed evidence**.

Đây là lý do cần 03b.1.


## 9A. Visualize L* transitions for the same A/B pair

Aggregate `edge_entropy` khó giải thích bằng lời.

Ta resolve timezone chỉ cho hai selected users, cluster lại ở complete-link 200 m, rồi count within-day L*→L* transitions.

Heatmap là case-level intuition; 03b/03b.1 aggregate metrics vẫn là evidence chính.


In [ ]:
selected_raw_stays = stays_user.loc[
    stays_user["user_id"].astype(str).isin([DEMO_A_USER, DEMO_B_USER])
].copy()
selected_resolved = base03a.resolve_stay_timezones(selected_raw_stays)
selected_resolved = selected_resolved.loc[selected_resolved["timezone_id"].notna()].copy()
selected_clustered, _ = base03a.cluster_behavior_locations(selected_resolved, 200.0)

def transition_matrix_for_user(user_id):
    user = selected_clustered.loc[
        selected_clustered["user_id"].astype(str).eq(str(user_id))
    ].sort_values("arrival_time_local", kind="stable").copy()
    edges = []
    for _, day in user.groupby("local_date", sort=True):
        seq = day.sort_values("arrival_time_local", kind="stable")["location_id"].astype(int).tolist()
        edges.extend((a,b) for a,b in zip(seq, seq[1:]) if a != b)
    if not edges:
        return pd.DataFrame(), pd.DataFrame(columns=["from_location","to_location","count"])
    edge_frame = pd.DataFrame(edges, columns=["from_location","to_location"])
    counts = edge_frame.value_counts([
        "from_location","to_location"
    ]).rename("count").reset_index().sort_values("count", ascending=False, kind="stable")
    labels = sorted(set(counts["from_location"]) | set(counts["to_location"]))
    matrix = pd.DataFrame(0, index=labels, columns=labels, dtype=int)
    for row in counts.itertuples(index=False):
        matrix.loc[int(row.from_location), int(row.to_location)] = int(row.count)
    matrix.index = [f"L{x}" for x in matrix.index]
    matrix.columns = [f"L{x}" for x in matrix.columns]
    return matrix, counts

for user_id, group_name in [(DEMO_A_USER,"A"), (DEMO_B_USER,"B")]:
    matrix, edges = transition_matrix_for_user(user_id)
    display(Markdown(f"### Group {group_name} — user `{user_id}`"))
    display(edges.head(12))
    if matrix.empty:
        print("No within-day transitions."); continue
    fig, ax = plt.subplots(figsize=(7,6))
    image = ax.imshow(matrix.to_numpy(), aspect="auto")
    ax.set_xticks(range(len(matrix.columns))); ax.set_xticklabels(matrix.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(matrix.index))); ax.set_yticklabels(matrix.index)
    ax.set(xlabel="to location", ylabel="from location", title=f"User {user_id} — L* transition counts")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = int(matrix.iloc[i,j])
            if value: ax.text(j, i, str(value), ha='center', va='center')
    fig.colorbar(image, ax=ax, label="transition count")
    plt.tight_layout(); plt.show()


### Link với 03b.1

Controlled population result support higher route diversity (`edge_entropy`) nhưng recurrent-edge difference bằng 0. Vì vậy cách đọc phù hợp là route complexity, không phải repeated distributed-work routes.


# Part IV — 03b.1 Observation-support-controlled audit

Câu hỏi mới:

**Nếu cho A và B cùng lượng usable observation exposure, khác biệt còn không?**

Với mỗi matched pair:

- downsample weekday A/B về cùng count;
- downsample weekend A/B về cùng count;
- route analysis làm riêng trên usable_for_motif days;
- bootstrap 500 lần;
- transport subset control theo matched labeled hours.

Sampling chỉ thay exposure, không thay frozen candidate definition.


In [ ]:
print("03b.1 support:", summary_03b1["support"])
print("Bootstrap repetitions:", summary_03b1["n_bootstraps"])

mobility_03b1 = load_csv(CACHE_03B1 / "mobility_summary.csv")
display(mobility_03b1.round(4))

x = np.arange(len(mobility_03b1))
y = mobility_03b1["paired_difference_median"].to_numpy()
low = y - mobility_03b1["ci95_low"].to_numpy()
high = mobility_03b1["ci95_high"].to_numpy() - y

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(x, y, yerr=np.vstack([low, high]), fmt="o", capsize=4)
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(mobility_03b1["metric"], rotation=45, ha="right")
ax.set_ylabel("Paired A - B difference")
ax.set_title("03b.1 — Mobility after equalizing exposure")
plt.tight_layout()
plt.show()


Mobility after exposure control:

- cleaned distance/day ≈ +22.88 km for A;
- movement proxy/day ≈ +0.74 h;
- boundary count/day ≈ +0.88;
- recurring-location count/day ≈ no difference;
- stay count/day ≈ no clear difference.

**Interpretation:** richer observation không giải thích hết movement magnitude.

Nhưng mobility metrics overlap với candidate construction, nên chưa phải independent semantic validation.


## 7. Independent route evidence sau exposure control

Đây là evidence stream quan trọng nhất.

Nếu route structure vẫn khác sau khi equalize days, difference không chỉ do A được observe nhiều hơn.


In [ ]:
route_03b1 = load_csv(CACHE_03B1 / "route_summary.csv")
display(route_03b1.round(4))

x = np.arange(len(route_03b1))
y = route_03b1["paired_difference_median"].to_numpy()
low = y - route_03b1["ci95_low"].to_numpy()
high = route_03b1["ci95_high"].to_numpy() - y

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(x, y, yerr=np.vstack([low, high]), fmt="o", capsize=4)
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(route_03b1["metric"], rotation=45, ha="right")
ax.set_ylabel("Paired A - B difference")
ax.set_title("03b.1 — Route evidence after equalizing exposure")
plt.tight_layout()
plt.show()

transport_03b1 = load_csv(CACHE_03B1 / "transport_summary.csv")
display(transport_03b1.round(4))
print("Eligible transport pairs:", summary_03b1["support"]["transport_pairs_eligible"])


Route result:

- edge entropy A-B ≈ +0.232;
- 95% bootstrap interval ≈ [0.006, 0.455];
- recurrent edges/day difference = 0;
- transition/day interval touches 0;
- distinct-edge/day interval touches 0.

**What survives:** route diversity/complexity.

**What does not survive as strong evidence:** recurrent route structure.

Transportation evidence chỉ có 1 eligible matched pair, nên không dùng cho population-level conclusion.


# Visual checkpoint — mentor demo flow

Nếu chỉ có 5–7 phút:

1. coverage funnel;
2. anchor population chart;
3. three real-user archetype maps;
4. sensitivity chart;
5. representative A/B maps;
6. A/B daily timelines;
7. transition heatmaps;
8. 03b.1 controlled CI charts;
9. final decision.


# Part V — Final research decision

| Stage | Question | Result | Decision |
|---|---|---|---|
| 03a | Behavior heterogeneous? | 72/136 multiple-anchor | Không assume Home+Office đủ cho mọi user |
| 03a schedule | Personalized schedule có evidence? | JSD không distinguish rõ khỏi null | Không personalize windows bằng JSD |
| 03a candidate | Có high-mobility OFFICE-abstained cohort? | 23 users | Tạo hypothesis, chưa semanticize |
| 03b | Cohort robust + independent evidence? | Robust, nhưng mixed + support imbalance | mixed evidence |
| 03b.1 | Support imbalance giải thích hết? | Không; movement + edge entropy còn | Freeze như mobility-complexity cohort |

Final interpretation:

23-user cohort
→ robust under threshold perturbation
→ not explained only by richer observation coverage
→ higher movement magnitude + route diversity
→ BUT no stronger recurrent-route evidence
→ transport labels too sparse
→ no occupation/work ground truth
→ robust descriptive mobility-complexity cohort
→ NOT validated mobile-work semantic regime.


## Production decision

Không thay đổi frozen Home/Office semantics từ 03a/03b/03b.1.

Giữ nguyên:

- CP1 cleaning/stay detection;
- CP2 v1 comparator;
- Home/Office production.

Lý do:

- chưa có Home/Office/work-role ground truth;
- recurrent route evidence chưa support hypothesis;
- transportation labels quá sparse;
- semantic leap từ mobility sang occupation chưa được validate.

Nếu sau này có external labels, 23-user cohort là candidate set tốt để validate future regime classifier.


# Mentor-ready talking points

### Nếu mentor hỏi: Em đã làm gì sau feedback?

Em không sửa heuristic ngay. Em tách feedback thành các hypothesis có thể falsify. Đầu tiên em audit heterogeneity toàn population. Sau đó em thấy một cohort 23 users nhiều anchors và mobility cao mà frozen OFFICE abstain toàn bộ. Em kiểm tra cohort qua sensitivity, matched controls, route structure và transport labels. Vì matched controls vẫn lệch observation support, em làm thêm pairwise exposure-controlled bootstrap. Sau control, movement magnitude và route diversity vẫn còn, nhưng recurrent route evidence và transport coverage không đủ. Vì vậy em giữ finding ở mức mobility-complexity cohort, chưa gọi là mobile-work và chưa sửa production semantics.

### Nếu mentor hỏi: Feedback mobile worker có đúng không?

Feedback đúng như một challenge đối với fixed-office assumption, nhưng dataset hiện tại chưa đủ evidence để xác nhận occupation/mobile-work semantics.

### Nếu mentor hỏi: Tại sao không tune thêm?

Vì tune thêm threshold trên cùng dataset dễ thành overfitting. Semanticization cần independent labels/evaluation, không chỉ thêm heuristic.


# Open follow-up — DBSCAN MinPts

Mentor note còn lại:

DBSCAN hiện dùng min_samples=1. Eps đã được sensitivity nhưng MinPts chưa được justify.

Với min_samples=1:

- mọi point tự nó là core;
- isolated stays trở thành singleton clusters;
- density-based noise gần như không tồn tại;
- connectivity/chaining permissive hơn.

Follow-up experiment:

- min_samples = 1 / 2 / 3 / 5;
- kết hợp representative eps values;
- đo recurring locations/users, noise rate, singleton behavior, p95/max diameter, clusters >200 m, users mất recurring-location support;
- compare với complete-link 200 m.

Decision rule: không chọn MinPts chỉ vì coverage cao nhất; cần explainable trade-off giữa coverage, density requirement, compactness và downstream support.


# Part II-A — Spatial case diagnostics with real GeoLife user IDs

## 5A. Chọn archetype users deterministic

Ta visualize dominant-anchor / two-anchor / multiple-anchor nhưng không chọn bằng mắt.

Rule: trong mỗi class, chọn user có `active_days` gần median của class nhất; tie-break bằng `user_id`.

Selected maps chỉ minh họa population finding, không thay aggregate evidence.


In [ ]:
def anchor_class_from_count(value):
    value = int(value)
    if value <= 0: return "no_stable_anchor"
    if value == 1: return "dominant_anchor"
    if value == 2: return "two_anchor"
    return "multiple_anchor"

case_pool = features_user.loc[
    pd.to_numeric(features_user["cp1_stay_count"], errors="coerce").fillna(0).gt(0)
].copy()
case_pool["anchor_class"] = (
    pd.to_numeric(case_pool["recurring_location_count"], errors="coerce")
    .fillna(0).astype(int).map(anchor_class_from_count)
)

selected_anchor_cases = []
for anchor_class in ["dominant_anchor", "two_anchor", "multiple_anchor"]:
    group = case_pool.loc[case_pool["anchor_class"].eq(anchor_class)].copy()
    median_days = group['active_days'].median()
    group["_distance_to_class_median"] = (group["active_days"] - median_days).abs()
    row = group.sort_values(
        ["_distance_to_class_median", "user_id"], kind="stable"
    ).iloc[0]
    selected_anchor_cases.append({
        "anchor_class": anchor_class,
        "user_id": str(row["user_id"]),
        "active_days": int(row["active_days"]),
        "cp1_stay_count": int(row["cp1_stay_count"]),
        "recurring_location_count": int(row["recurring_location_count"]),
    })

selected_anchor_cases = pd.DataFrame(selected_anchor_cases)
display(selected_anchor_cases)


## 5B. Interactive map helper

- small points = individual CP1 stays;
- L0/L1/... markers = recurring location clusters with at least 2 stays;
- line = chronological stay-to-stay path;
- popup = raw GeoLife user ID, location ID, stay count, dwell.

`L*` remains an abstract recurring-location label, not HOME/OFFICE semantics.


In [ ]:
LOCATION_COLORS = ["red","blue","green","purple","orange","darkred","darkblue","darkgreen","cadetblue","pink"]

def user_location_summary(user_stays):
    rows = []
    for location_id, group in user_stays.groupby("location_id", sort=True):
        rows.append({
            "location_id": int(location_id),
            "latitude": float(group["latitude"].median()),
            "longitude": float(group["longitude"].median()),
            "stay_count": int(len(group)),
            "is_recurring": bool(len(group) >= 2),
            "dwell_h": float(group["duration_s"].sum() / 3600),
        })
    return pd.DataFrame(rows)

def build_user_map(user_id, title=None):
    user_id = str(user_id)
    user_stays = clustered_stays.loc[
        clustered_stays["user_id"].astype(str).eq(user_id)
    ].sort_values("arrival_time_utc", kind="stable").copy()
    if user_stays.empty:
        raise ValueError(f"No cached stays for user {user_id}")
    locations = user_location_summary(user_stays)
    center = [float(user_stays["latitude"].median()), float(user_stays["longitude"].median())]
    fmap = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap", control_scale=True)
    plugins.Fullscreen().add_to(fmap)
    path_layer = folium.FeatureGroup(name="Chronological stay path", show=True)
    point_layer = folium.FeatureGroup(name="Individual CP1 stays", show=True)
    location_layer = folium.FeatureGroup(name="Recurring L* locations", show=True)
    coords = user_stays[["latitude","longitude"]].to_numpy().tolist()
    if len(coords) >= 2:
        folium.PolyLine(coords, weight=2, opacity=0.55).add_to(path_layer)
    for row in user_stays.itertuples(index=False):
        loc = int(row.location_id)
        color = LOCATION_COLORS[loc % len(LOCATION_COLORS)]
        folium.CircleMarker(
            location=[float(row.latitude), float(row.longitude)],
            radius=3, color=color, fill=True, fill_opacity=0.45,
            tooltip=f"user={user_id} | L{loc} | {pd.Timestamp(row.arrival_time_utc)}",
        ).add_to(point_layer)
    for row in locations.itertuples(index=False):
        if not row.is_recurring:
            continue
        color = LOCATION_COLORS[int(row.location_id) % len(LOCATION_COLORS)]
        popup = (
            f"<b>User {user_id} — L{int(row.location_id)}</b><br>"
            f"stays: {int(row.stay_count)}<br>dwell: {row.dwell_h:.1f} h"
        )
        folium.Marker(
            [row.latitude, row.longitude],
            tooltip=f"User {user_id} — L{int(row.location_id)}",
            popup=folium.Popup(popup, max_width=320),
            icon=folium.Icon(color=color, icon="info-sign"),
        ).add_to(location_layer)
    path_layer.add_to(fmap)
    point_layer.add_to(fmap)
    location_layer.add_to(fmap)
    folium.LayerControl(collapsed=False).add_to(fmap)
    export_path = MAP_EXPORT_DIR / f"user_{user_id}_spatial_case.html"
    fmap.save(export_path)
    return fmap, locations, export_path


## 5C. Three archetype maps

Các map dưới đây cho mentor nhìn trực tiếp dominant / two-anchor / multiple-anchor behavior.


In [ ]:
for case in selected_anchor_cases.itertuples(index=False):
    display(Markdown(
        f"### {case.anchor_class} — GeoLife user `{case.user_id}`  \n"
        f"`active_days={case.active_days}`, `CP1 stays={case.cp1_stay_count}`, "
        f"`recurring locations={case.recurring_location_count}`"
    ))
    fmap, location_table, export_path = build_user_map(
        case.user_id, title=case.anchor_class
    )
    display(location_table.round({'dwell_h': 1}))
    display(fmap)
    print("HTML export:", export_path)


### Cách đọc map archetypes

Map support rằng recurring-location structure thật sự heterogeneous.

Map không support `L0 = HOME`, `L1 = OFFICE`, hay `multiple-anchor = mobile worker`.
Semantic interpretation vẫn cần evidence riêng.


## 8A. A real matched A/B pair — map + metrics

Pair demo không chọn bằng mắt.

Rule: chọn Group A candidate có `edge_entropy` gần median Group A nhất, sau đó lấy đúng matched Group B control.

Case này minh họa population result; nó không thay 23-pair evidence.


In [ ]:
route_a = route_user.loc[route_user["group"].eq("A")].copy()
route_a["edge_entropy"] = pd.to_numeric(route_a["edge_entropy"], errors="coerce")
median_entropy_a = route_a["edge_entropy"].median()
route_a["_median_distance"] = (route_a["edge_entropy"] - median_entropy_a).abs()
eligible_a = route_a.loc[
    route_a["user_id"].isin(set(matched_user["candidate_user_id"]))
].sort_values(["_median_distance","user_id"], kind="stable")
DEMO_A_USER = str(eligible_a.iloc[0]["user_id"])
DEMO_B_USER = str(matched_user.loc[
    matched_user["candidate_user_id"].eq(DEMO_A_USER), "control_user_id"
].iloc[0])
print("Representative Group A user:", DEMO_A_USER)
print("Matched Group B control:", DEMO_B_USER)
print("Group A median edge entropy:", round(float(median_entropy_a), 3))

pair_features = features_user.loc[
    features_user["user_id"].isin([DEMO_A_USER, DEMO_B_USER]),
    ["user_id","active_days","usable_temporal_days","usable_active_days","cp1_stay_count",
     "recurring_location_count","distance_per_usable_day_km","movement_proxy_per_usable_day_h",
     "boundary_count_per_usable_day"],
].copy()
pair_route = route_user.loc[
    route_user["user_id"].isin([DEMO_A_USER, DEMO_B_USER]),
    ["user_id","group","transition_count","distinct_edge_count","recurrent_edge_count",
     "top_edge_frequency","edge_entropy"],
].copy()
display(pair_features.round(3))
display(pair_route.round(3))


In [ ]:
display(Markdown(f"### Group A candidate — user `{DEMO_A_USER}`"))
map_a, locations_a, export_a = build_user_map(DEMO_A_USER, "Group A candidate")
display(locations_a.round({'dwell_h': 1}))
display(map_a)
print("HTML export:", export_a)

display(Markdown(f"### Matched Group B control — user `{DEMO_B_USER}`"))
map_b, locations_b, export_b = build_user_map(DEMO_B_USER, "Group B control")
display(locations_b.round({'dwell_h': 1}))
display(map_b)
print("HTML export:", export_b)


### Cách đọc pair map

Map là intuition layer. Không kết luận A mobile hơn chỉ bằng mắt; 03b.1 mới control observation exposure.


## 8B. Daily mobility timeline của đúng A/B pair

Timeline dùng frozen cleaned-point daily cache, không đọc raw `.plt`.
Nó cho thấy support density và movement over time — lý do observation imbalance phải được audit tiếp ở 03b.1.


In [ ]:
pair_days = point_days_behavior.loc[
    point_days_behavior["user_id"].astype(str).isin([DEMO_A_USER, DEMO_B_USER])
].copy()

for user_id, group_name in [(DEMO_A_USER, "A"), (DEMO_B_USER, "B")]:
    user_days = pair_days.loc[
        pair_days["user_id"].astype(str).eq(user_id)
    ].sort_values("local_date", kind="stable").copy()
    display(Markdown(f"### Daily mobility — Group {group_name}, user `{user_id}`"))
    fig, ax = plt.subplots(figsize=(11,4))
    ax.plot(pd.to_datetime(user_days["local_date"]), user_days["cleaned_distance_km"], marker=".")
    ax.set(title=f"User {user_id} — daily cleaned distance", xlabel="local date", ylabel="km/day")
    plt.tight_layout(); plt.show()
    fig, ax = plt.subplots(figsize=(11,4))
    ax.plot(pd.to_datetime(user_days["local_date"]), user_days["movement_duration_proxy_h"], marker=".")
    ax.set(title=f"User {user_id} — movement-duration proxy", xlabel="local date", ylabel="hours/day")
    plt.tight_layout(); plt.show()
    fig, ax = plt.subplots(figsize=(11,3))
    ax.bar(pd.to_datetime(user_days["local_date"]), user_days["boundary_count"])
    ax.set(title=f"User {user_id} — quality/continuity boundaries", xlabel="local date", ylabel="boundaries/day")
    plt.tight_layout(); plt.show()


### Vì sao timeline dẫn tới 03b.1?

Nếu A có nhiều observed days hơn B, lifetime transition metrics có thể inflated. Vì vậy 03b.1 downsample từng pair về matched weekday/weekend exposure rồi bootstrap lại.


# Final takeaway

**EDA mới không chứng minh mobile workers. Nó chứng minh current fixed-office view không mô tả hết behavioral diversity, và cô lập được một robust mobility-complexity cohort đáng giữ lại cho future validation.**

Research insight → giữ.

Semantic label → chưa.

Production change → chưa.
